# **BioPred Phase 0 -- Activity Evidence Extraction**

This notebook extracts the long-format ChEMBL activity evidence table for the provisional BioPred GABA-A target set identified in `01_chembl_discovery.ipynb`.

Notebook 01 handled target discovery, human GABA-A refinement, endpoint availability checks, and early data-quality risks. This notebook moves from discovery into activity-record extraction.

The goal is not to create a final model-ready dataset. The goal is to produce a raw, auditable evidence table that preserves target, assay, endpoint, molecule, activity, and provenance metadata for later cleaning.

## Phase context

This notebook is part of **BioPred Phase 0**.

It bridges:

- target/endpoint discovery, and
- curated evidence dataset construction.

## Expected table grain

The initial evidence table should use:

`activity_id`

Each row should represent one ChEMBL activity record. Duplicate handling, endpoint weighting, pX transformation, activity labeling, and model-ready aggregation are deferred.

## Notebook Objectives

1. Load the provisional BioPred GABA-A target set from the Notebook 01 artifact.
2. Extract raw IC50, EC50, and Ki activity records from ChEMBL at `activity_id` grain.
3. Preserve target, assay, endpoint, molecule, structure, and basic source metadata needed for downstream cleaning.
4. Audit extraction totals, endpoint distributions, units, relations, missingness, unique molecules, and structure availability.
5. Save the raw activity evidence table as an interim Parquet artifact for Notebook 03.

In [1]:
# list imports needed for this notebook
from pathlib import Path
import sys
import os
from dotenv import load_dotenv
import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

# Force notebook runtime to project root
%cd /home/ryanm/projects/BioPred

# define paths and link src.paths
SRC_DIR = Path.cwd() / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import biopred.paths as paths

load_dotenv(paths.PROJECT_ROOT / ".env", override=True)

# create the database engine
db_url = URL.create(
    drivername = os.getenv("BIOPRED_DB_DIALECT"),
    username=os.getenv("BIOPRED_DB_USER"),
    password=os.getenv("BIOPRED_DB_PASSWORD"),
    host=os.getenv("BIOPRED_DB_HOST"),
    port=int(os.getenv("BIOPRED_DB_PORT")),
    database=os.getenv("BIOPRED_DB_NAME")
)

engine = create_engine(db_url, pool_pre_ping=True)
schema = os.getenv("BIOPRED_DB_SCHEMA", "public")

/home/ryanm/projects/BioPred



We will now load our previous target set, which was saved as a .csv file in Notebook 01.  Let's recall this as we will use it throughout this notebook.

In [2]:
# Load our saved artifact from our data/interim folder.
provisional_gabaa_targets = pd.read_csv(
    paths.INTERIM_DATA_DIR / "provisional_gabaa_targets.csv"
)

# Creating the list of target_ids to use in our next section
provisional_target_ids = provisional_gabaa_targets["target_chembl_id"].tolist()

provisional_gabaa_targets

,tid,target_chembl_id,pref_name,organism,target_type
0,105087,CHEMBL2111392,GABA A receptor alpha-1/beta-1/gamma-2,Homo sapiens,PROTEIN COMPLEX
1,104921,CHEMBL2111413,GABA A receptor alpha-2/beta-2/gamma-2,Homo sapiens,PROTEIN COMPLEX
2,105038,CHEMBL2111339,GABA A receptor alpha-3/beta-2/gamma-2,Homo sapiens,PROTEIN COMPLEX
3,105121,CHEMBL2111366,GABA A receptor alpha-4/beta-3/gamma-2,Homo sapiens,PROTEIN COMPLEX
4,105059,CHEMBL2111370,GABA A receptor alpha-6/beta-2/gamma-2,Homo sapiens,PROTEIN COMPLEX
5,117721,CHEMBL4106151,GABA-A receptor alpha-1/beta-3,Homo sapiens,PROTEIN COMPLEX
6,117727,CHEMBL4106157,GABA-A receptor alpha-4/beta-3/delta,Homo sapiens,PROTEIN COMPLEX
7,104291,CHEMBL1907597,GABA-A receptor; GABA-A site (alpha1/beta2 int...,Homo sapiens,PROTEIN COMPLEX
8,104905,CHEMBL2109244,GABA-A receptor; agonist GABA site,Homo sapiens,PROTEIN COMPLEX GROUP
9,104757,CHEMBL2095172,GABA-A receptor; alpha-1/beta-2/gamma-2,Homo sapiens,PROTEIN COMPLEX



Now we will look to extract more material evidence for our acquired list of target_ids.  We will not look to clean, deduplicate, or filter here as it is understood we are still in discovery mode.  The goal still is to acquire more evidence from our ChEMBL database.  Some of the tables I will look to extract from are:

- target_dictionary
- assays
- activities
- compound_structures
- molecule_dictionary

Which were all pointed out as being preferred tables in our Notebook 01 previously.  This will just be a broad extraction at this time.

In [3]:
activity_evidence_sql = """
SELECT
    act.activity_id,

    td.chembl_id AS target_chembl_id,
    td.pref_name AS target_pref_name,
    td.target_type,
    td.organism,

    a.assay_id,
    a.chembl_id AS assay_cheml_id,
    a.assay_type,
    a.confidence_score,

    act.molregno,
    md.chembl_id AS molecule_chembl_id,
    cs.canonical_smiles,
    cs.standard_inchi_key,
    md.molecule_type,
    md.max_phase,

    act.standard_type,
    act.standard_relation,
    act.standard_value,
    act.standard_units,
    act.pchembl_value,
    act.doc_id
FROM target_dictionary td
JOIN assays a
    ON td.tid = a.tid
JOIN activities act
    ON a.assay_id = act.assay_id
JOIN molecule_dictionary md
    ON act.molregno = md.molregno
LEFT JOIN compound_structures cs
    ON md.molregno = cs.molregno
WHERE td.chembl_id = ANY(%(target_ids)s)
    AND act.standard_type IN ('IC50', 'EC50', 'Ki')
ORDER BY
    td.chembl_id,
    act.standard_type,
    act.activity_id;
"""

activity_evidence_raw = pd.read_sql_query(
    activity_evidence_sql,
    con=engine,
    params={"target_ids" : provisional_target_ids},
)

activity_evidence_raw

,activity_id,target_chembl_id,target_pref_name,target_type,organism,assay_id,assay_cheml_id,assay_type,confidence_score,molregno,...,canonical_smiles,standard_inchi_key,molecule_type,max_phase,standard_type,standard_relation,standard_value,standard_units,pchembl_value,doc_id
0,6308927,CHEMBL1907597,GABA-A receptor; GABA-A site (alpha1/beta2 int...,PROTEIN COMPLEX,Homo sapiens,761260,CHEMBL1816618,B,6,1169334,...,COc1ccc(-c2cc(CC(O)CO)ccc2O)cc1CC(O)CO,LOZQPRNAFFRXNY-UHFFFAOYSA-N,Small molecule,NaN,EC50,None,NaN,None,NaN,58535
1,6308928,CHEMBL1907597,GABA-A receptor; GABA-A site (alpha1/beta2 int...,PROTEIN COMPLEX,Homo sapiens,761260,CHEMBL1816618,B,6,1169353,...,C=CCc1cc(-c2ccc(OC)c(CC=C)c2)c(O)c(N(CC)CC)c1,CGLDEJWECRQQQF-UHFFFAOYSA-N,Small molecule,NaN,EC50,None,NaN,None,NaN,58535
2,6308929,CHEMBL1907597,GABA-A receptor; GABA-A site (alpha1/beta2 int...,PROTEIN COMPLEX,Homo sapiens,761260,CHEMBL1816618,B,6,1169346,...,C=CCc1ccc(OCCCCC)c(-c2ccc(OCCCCC)c(CC=C)c2)c1,GLCRBBLELTVRAZ-UHFFFAOYSA-N,Small molecule,NaN,EC50,None,NaN,None,NaN,58535
3,6308930,CHEMBL1907597,GABA-A receptor; GABA-A site (alpha1/beta2 int...,PROTEIN COMPLEX,Homo sapiens,761260,CHEMBL1816618,B,6,1169345,...,C=CCc1ccc(O)c(-c2ccc(OCCCCC)c(CC=C)c2)c1,WIQKQADUKUIMKN-UHFFFAOYSA-N,Small molecule,NaN,EC50,None,NaN,None,NaN,58535
4,6308931,CHEMBL1907597,GABA-A receptor; GABA-A site (alpha1/beta2 int...,PROTEIN COMPLEX,Homo sapiens,761260,CHEMBL1816618,B,6,1169344,...,C=CCc1ccc(OCCCCC)c(-c2ccc(O)c(CC=C)c2)c1,ZYXPZNUJASUZOF-UHFFFAOYSA-N,Small molecule,NaN,EC50,None,NaN,None,NaN,58535
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5574,6308925,CHEMBL4106151,GABA-A receptor alpha-1/beta-3,PROTEIN COMPLEX,Homo sapiens,761264,CHEMBL1816622,B,6,18078,...,C=CCc1ccc(O)c(-c2ccc(O)c(CC=C)c2)c1,FVYXIJYOAGAUQK-UHFFFAOYSA-N,Small molecule,NaN,EC50,=,59600.0,nM,4.22,58535
5575,22478795,CHEMBL4106151,GABA-A receptor alpha-1/beta-3,PROTEIN COMPLEX,Homo sapiens,2032692,CHEMBL4686850,B,7,2495760,...,CC(C)c1cc(C2(C(F)(F)F)N=N2)cc(C(C)C)c1CO,FGQXQCSYLCZOIU-UHFFFAOYSA-N,Unknown,NaN,EC50,=,2900.0,nM,5.54,117894
5576,12563721,CHEMBL4106151,GABA-A receptor alpha-1/beta-3,PROTEIN COMPLEX,Homo sapiens,924909,CHEMBL3073012,B,7,164913,...,N#Cc1nn(-c2c(Cl)cc(C(F)(F)F)cc2Cl)c(N)c1[S+]([...,ZOCSXAVNDGMNBV-UHFFFAOYSA-N,Small molecule,-1.0,IC50,=,28.0,nM,7.55,71002
5577,12563722,CHEMBL4106151,GABA-A receptor alpha-1/beta-3,PROTEIN COMPLEX,Homo sapiens,924909,CHEMBL3073012,B,7,16654,...,Cl[C@H]1[C@H](Cl)[C@@H](Cl)[C@@H](Cl)[C@H](Cl)...,JLYXXMFPNIAWKQ-GNIYUCBRSA-N,Small molecule,4.0,IC50,=,12.0,nM,7.92,71002


##### Now let's take a look at this new data and see what it might tell us.

In [4]:
# Check columns
activity_evidence_raw.columns

Index(['activity_id', 'target_chembl_id', 'target_pref_name', 'target_type',
       'organism', 'assay_id', 'assay_cheml_id', 'assay_type',
       'confidence_score', 'molregno', 'molecule_chembl_id',
       'canonical_smiles', 'standard_inchi_key', 'molecule_type', 'max_phase',
       'standard_type', 'standard_relation', 'standard_value',
       'standard_units', 'pchembl_value', 'doc_id'],
      dtype='object')

In [5]:
# Inspect first few rows
activity_evidence_raw.head()

,activity_id,target_chembl_id,target_pref_name,target_type,organism,assay_id,assay_cheml_id,assay_type,confidence_score,molregno,...,canonical_smiles,standard_inchi_key,molecule_type,max_phase,standard_type,standard_relation,standard_value,standard_units,pchembl_value,doc_id
0,6308927,CHEMBL1907597,GABA-A receptor; GABA-A site (alpha1/beta2 int...,PROTEIN COMPLEX,Homo sapiens,761260,CHEMBL1816618,B,6,1169334,...,COc1ccc(-c2cc(CC(O)CO)ccc2O)cc1CC(O)CO,LOZQPRNAFFRXNY-UHFFFAOYSA-N,Small molecule,NaN,EC50,None,NaN,None,NaN,58535
1,6308928,CHEMBL1907597,GABA-A receptor; GABA-A site (alpha1/beta2 int...,PROTEIN COMPLEX,Homo sapiens,761260,CHEMBL1816618,B,6,1169353,...,C=CCc1cc(-c2ccc(OC)c(CC=C)c2)c(O)c(N(CC)CC)c1,CGLDEJWECRQQQF-UHFFFAOYSA-N,Small molecule,NaN,EC50,None,NaN,None,NaN,58535
2,6308929,CHEMBL1907597,GABA-A receptor; GABA-A site (alpha1/beta2 int...,PROTEIN COMPLEX,Homo sapiens,761260,CHEMBL1816618,B,6,1169346,...,C=CCc1ccc(OCCCCC)c(-c2ccc(OCCCCC)c(CC=C)c2)c1,GLCRBBLELTVRAZ-UHFFFAOYSA-N,Small molecule,NaN,EC50,None,NaN,None,NaN,58535
3,6308930,CHEMBL1907597,GABA-A receptor; GABA-A site (alpha1/beta2 int...,PROTEIN COMPLEX,Homo sapiens,761260,CHEMBL1816618,B,6,1169345,...,C=CCc1ccc(O)c(-c2ccc(OCCCCC)c(CC=C)c2)c1,WIQKQADUKUIMKN-UHFFFAOYSA-N,Small molecule,NaN,EC50,None,NaN,None,NaN,58535
4,6308931,CHEMBL1907597,GABA-A receptor; GABA-A site (alpha1/beta2 int...,PROTEIN COMPLEX,Homo sapiens,761260,CHEMBL1816618,B,6,1169344,...,C=CCc1ccc(OCCCCC)c(-c2ccc(O)c(CC=C)c2)c1,ZYXPZNUJASUZOF-UHFFFAOYSA-N,Small molecule,NaN,EC50,None,NaN,None,NaN,58535


In [6]:
# Verify endpoint counts
activity_evidence_raw["standard_type"].value_counts(dropna=False)

standard_type
Ki      4321
EC50     677
IC50     581
Name: count, dtype: int64

In [7]:
# Look at unique number of molecules
activity_evidence_raw["molecule_chembl_id"].nunique()

1960

In [8]:
# Look for missing structures
activity_evidence_raw["canonical_smiles"].isna().mean()

np.float64(0.0)

In [9]:
# Checking our standard_units distribution
activity_evidence_raw["standard_units"].value_counts(dropna=False)

standard_units
nM        5473
None       103
%            2
microA       1
Name: count, dtype: int64

In [10]:
# Checking the standard_relation counts
activity_evidence_raw["standard_relation"].value_counts(dropna=False)

standard_relation
=       4703
>        724
None     148
<          2
~          1
>=         1
Name: count, dtype: int64

In [12]:
# Save as an artifact for later
activity_evidence_path = paths.INTERIM_DATA_DIR / "gabaa_activity_evidence_raw.parquet"

activity_evidence_raw.to_parquet(activity_evidence_path, index=False)

# Verify creation and path
print(activity_evidence_path)
print(activity_evidence_raw.shape)

/home/ryanm/projects/BioPred/data/interim/gabaa_activity_evidence_raw.parquet
(5579, 21)


In [13]:
# Creating a summary table with our findings from above for later use
extraction_summary = pd.DataFrame(
    {
        "metric": [
            "activity_records",
            "unique_molecules",
            "unique_targets",
            "unique_assays",
            "missing_canonical_smiles_rate",
            "nM_rows",
            "exact_relation_rows",
        ],
        "value": [
            len(activity_evidence_raw),
            activity_evidence_raw["molecule_chembl_id"].nunique(),
            activity_evidence_raw["target_chembl_id"].nunique(),
            activity_evidence_raw["assay_id"].nunique(),
            activity_evidence_raw["canonical_smiles"].isna().mean(),
            (activity_evidence_raw["standard_units"] == "nM").sum(),
            (activity_evidence_raw["standard_relation"] == "=").sum(),
        ],
    }
)

extraction_summary

,metric,value
0,activity_records,5579.0
1,unique_molecules,1960.0
2,unique_targets,15.0
3,unique_assays,518.0
4,missing_canonical_smiles_rate,0.0
5,nM_rows,5473.0
6,exact_relation_rows,4703.0


### Notebook Conclusions

This notebook extracted the raw long-format ChEMBL activity evidence table for the provisional BioPred GABA-A target set.

The extraction produced 5,579 activity records across 1,960 unique molecules. The endpoint row counts matched the endpoint coverage totals from `01_chembl_discovery.ipynb`, confirming that the extraction query is consistent with the discovery audit.

All extracted records have canonical SMILES available, supporting downstream RDKit featurization. Most records use standardized `nM` units, while a small number have missing or incompatible units. Relation symbols are mostly exact (`=`), but a meaningful subset use `>` relations, which will require explicit handling during cleaning and label construction.

The raw activity evidence table was saved as an interim artifact for downstream cleaning and viability assessment.

